# 02 — Tái lập lựa chọn danh mục

**Mục tiêu:** đối chiếu toàn bộ targets và universe audit của năm strategy ở 60 decisions Dev và 24 decisions Holdout.

**Đầu vào:** cùng pin của Notebook 01. **Đầu ra:** bảng kiểm tra, minh họa ứng viên/điểm/target và `02_verification.json`. Không sinh candidate/ranking CSV canonical mới.

`Strategies` và `ranking` chính thức thực hiện tính điểm. Notebook không viết lại thuật toán chọn mã.

In [1]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "M3/sources.json").is_file())
sys.path.insert(0, str(root / "M3"))
from m3c import Context, table, NAMES
ctx = Context()
print("Code nguồn:", ctx.identity["source_commit"])
print("Code thực thi:", ctx.identity["execution_commit"])
print("Config và nguồn canonical: PASS")
print("Evidence:", ctx.output)

Code nguồn: c9c1f5f0c88f2dbb3ebbb50effa47003b24b66cf
Code thực thi: c9c1f5f0c88f2dbb3ebbb50effa47003b24b66cf
Config và nguồn canonical: PASS
Evidence: C:\Users\HP\Downloads\Phân cụm động lượng TTCK\M3\reproduction\20261104T145912988868Z


## 1. Tái lập đủ hai đoạn và năm strategy

Đối chiếu từng trường, từng dòng của target và audit. Identifiers/status/order phải khớp chính xác; số thực dùng rtol=1e-9, atol=1e-10. Một discrepancy làm notebook dừng.

In [2]:
summary, periods = ctx.selection()
table(summary)
print("Số strategy/segment đã đối chiếu:", len(summary))

segment,strategy,decisions,targets_rows,audit_rows,result
development,Combined,60,787,45646,PASS
development,Market-only,60,1595,45646,PASS
development,Financial-only,60,4012,45646,PASS
development,Equal-weight,60,25647,45646,PASS
development,Momentum-only,60,7722,45646,PASS
holdout,Combined,24,259,30155,PASS
holdout,Market-only,24,828,30155,PASS
holdout,Financial-only,24,1706,30155,PASS
holdout,Equal-weight,24,18087,30155,PASS
holdout,Momentum-only,24,5439,30155,PASS


Số strategy/segment đã đối chiếu: 10


## 2. Tập ứng viên và số mã được chọn của Combined

N_selected là số mã mục tiêu, chưa phải số mã thực giữ sau giao dịch. Kỳ dưới N_min không tạo mua mới. Kỳ cuối vẫn lưu target nhưng không execution ngoài segment.

In [3]:
table([r for r in periods if r["strategy"] == "combined"])

segment,strategy,decision_date,candidates,N_selected,status
development,combined,2019-01-31,63,19,OK
development,combined,2019-02-28,68,21,OK
development,combined,2019-03-29,4,0,UNIVERSE_TOO_SMALL
development,combined,2019-04-26,8,0,UNIVERSE_TOO_SMALL
development,combined,2019-05-31,15,5,OK
development,combined,2019-06-28,15,5,OK
development,combined,2019-07-31,14,5,OK
development,combined,2019-08-30,11,4,OK
development,combined,2019-09-30,22,7,OK
development,combined,2019-10-31,12,4,OK


## 3. Ví dụ sáu tiêu chí, điểm và tỷ trọng mục tiêu

Ba MC: mom_126 cao, downside_vol_63 thấp, liquidity_21 cao. Ba FC: roe/f_score/z_score cao. Score là trung bình percentile theo nhóm, Combined cân bằng hai nhóm 50/50. Thiếu một phần tiêu chí dùng 0,5 và có flag; thiếu toàn bộ FC bị loại trước N_min.

Các bảng sau minh họa kỳ đầu mỗi đoạn, gồm một số mã được chọn và không được chọn. Điểm/số liệu lấy từ kết quả module chính thức. Giá trị tỷ trọng 0,05 nghĩa là 5%.

In [4]:
for segment in ctx.runs:
    example = ctx.ranking_example(segment)
    visible = [r for r in example if r["selected"]][:8] + [r for r in example if not r["selected"]][:5]
    print(segment, ctx.decisions(segment)[0])
    table(visible, ["ticker", "mom_126", "downside_vol_63", "liquidity_21", "roe", "f_score", "z_score", "flags"])
    table(visible, ["ticker", "pct_mom_126", "pct_downside_vol_63", "pct_liquidity_21", "pct_roe", "pct_f_score", "pct_z_score", "S_market", "S_fin", "combined_score", "rank", "selected", "target_weight"])

development 2019-01-31


ticker,mom_126,downside_vol_63,liquidity_21,roe,f_score,z_score,flags
NNC,0.20327787,0.10159131,6.102381e+08,0.43610991,,10.092908,f_score
DSN,0.17873738,0.084510229,3.3975238e+08,0.39839753,,46.409378,f_score
LIX,0.2109317,0.11393366,1.1654471e+09,0.31983437,,6.7237653,f_score
HMC,0.28863615,0.10642615,1.5159433e+09,0.22074554,,3.6236556,f_score
AMV,1.3486909,0.25329453,1.9747315e+09,0.24568886,,6.2754029,f_score
DHA,0.157327,0.13234853,1.0369524e+09,0.16819737,,14.187729,f_score
TV2,0.19210424,0.21059127,3.2842502e+09,0.53323178,6,2.8490179,
GHC,0.18713532,0.15274029,84931357,0.36065311,,7.4399273,f_score
CLC,0.085318782,0.18127127,1.6247619e+08,0.25758886,,5.3700387,f_score
CMX,2.3877523,0.44212041,1.6314762e+09,0.64766686,,1.0595794,f_score


ticker,pct_mom_126,pct_downside_vol_63,pct_liquidity_21,pct_roe,pct_f_score,pct_z_score,S_market,S_fin,combined_score,rank,selected,target_weight
NNC,0.75806452,0.93548387,0.64516129,0.90322581,0.5,0.93333333,0.77956989,0.77885305,0.77921147,1,True,0.052631579
DSN,0.69354839,0.98387097,0.58064516,0.85483871,0.5,0.98333333,0.75268817,0.77939068,0.76603943,2,True,0.052631579
LIX,0.79032258,0.88709677,0.74193548,0.75806452,0.5,0.78333333,0.80645161,0.68046595,0.74345878,3,True,0.052631579
HMC,0.87096774,0.90322581,0.79032258,0.4516129,0.5,0.5,0.85483871,0.48387097,0.66935484,4,True,0.052631579
AMV,0.98387097,0.25806452,0.83870968,0.61290323,0.5,0.76666667,0.69354839,0.6265233,0.66003584,5,True,0.052631579
DHA,0.66129032,0.79032258,0.72580645,0.29032258,0.5,0.95,0.72580645,0.58010753,0.65295699,6,True,0.052631579
TV2,0.74193548,0.37096774,0.90322581,0.9516129,0.64285714,0.28333333,0.67204301,0.62593446,0.64898874,7,True,0.052631579
GHC,0.70967742,0.67741935,0.29032258,0.80645161,0.5,0.85,0.55913978,0.7188172,0.63897849,8,True,0.052631579
CLC,0.61290323,0.5,0.41935484,0.66129032,0.5,0.68333333,0.51075269,0.61487455,0.56281362,,False,
CMX,1,0,0.82258065,1,0.5,0.033333333,0.60752688,0.51111111,0.559319,,False,


holdout 2024-01-31


ticker,mom_126,downside_vol_63,liquidity_21,roe,f_score,z_score,flags
DGC,0.21997259,0.14048383,1.8726952e+11,0.67340848,6,9.1547368,
PNJ,0.13740329,0.087001772,1.2427302e+11,0.25049814,6,6.7512795,
DCM,0.045300418,0.16316189,8.5289143e+10,0.47881048,9,5.6000712,
FPT,0.15678697,0.11305484,2.0520051e+11,0.27240118,6,3.6659892,
CSV,0.4506719,0.16919445,2.6593429e+10,0.28639736,,5.9278086,f_score
LIX,0.29724754,0.1013767,2.9167143e+09,0.26696007,7,5.6980092,
FOX,0.41488885,0.08954352,4.5323661e+09,0.31606823,6,2.5111635,
BMP,0.23680404,0.20790781,1.7840363e+10,0.28252206,9,10.800512,
DDV,-0.0027700531,0.15794763,7.3923548e+09,0.2193572,,5.4429052,f_score
SBH,0.66475505,0.56373339,1.4184445e+08,0.30202581,,17.438265,f_score


ticker,pct_mom_126,pct_downside_vol_63,pct_liquidity_21,pct_roe,pct_f_score,pct_z_score,S_market,S_fin,combined_score,rank,selected,target_weight
DGC,0.59375,0.73958333,0.89583333,0.96875,0.58441558,0.94791667,0.74305556,0.83369408,0.78837482,1,True,0.033333333
PNJ,0.48958333,0.96875,0.875,0.75,0.58441558,0.92708333,0.77777778,0.75383297,0.76580538,2,True,0.033333333
DCM,0.30208333,0.59375,0.78125,0.94791667,0.94155844,0.86458333,0.55902778,0.91801948,0.73852363,3,True,0.033333333
FPT,0.51041667,0.88541667,0.90625,0.79166667,0.58441558,0.6875,0.76736111,0.68786075,0.72761093,4,True,0.033333333
CSV,0.875,0.54166667,0.65625,0.84375,0.5,0.90625,0.69097222,0.75,0.72048611,5,True,0.033333333
LIX,0.65625,0.92708333,0.34375,0.78125,0.71428571,0.875,0.64236111,0.79017857,0.71626984,6,True,0.033333333
FOX,0.84375,0.95833333,0.42708333,0.875,0.58441558,0.57291667,0.74305556,0.67744408,0.71024982,7,True,0.033333333
BMP,0.60416667,0.34375,0.58333333,0.8125,0.94155844,0.96875,0.51041667,0.90760281,0.70900974,8,True,0.033333333
DDV,0.25,0.61458333,0.5,0.69791667,0.5,0.82291667,0.45486111,0.67361111,0.56423611,,False,
SBH,0.92708333,0.020833333,0.083333333,0.86458333,0.5,0.98958333,0.34375,0.78472222,0.56423611,,False,


## 4. Kết luận

Target khớp không đồng nghĩa lệnh mua thành công. Bước execution và phần danh mục thực giữ được kiểm chứng ở Notebook 03. Không chọn lại rule từ lợi suất Dev/Holdout.

In [5]:
print(ctx.finish())
print("NOTEBOOK_02_PASS")

PASS — nguồn canonical giữ nguyên; không có Final performance.
NOTEBOOK_02_PASS
